# Cross-validate 4 models, pick winner for tunning

In [26]:
# imports 
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay,
)

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

## 1. Load data + recap baseline

In [35]:
df = pd.read_csv("../data/processed/cleaned.csv")
df.head(2)

,SeniorCitizen,Partner,Dependents,tenure,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
0,No,Yes,No,1,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,No
1,No,No,No,34,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,No


In [52]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [53]:
y_train = y_train.map({"No": 0, "Yes":1 })
y_test = y_test.map({"No": 0, "Yes":1 })

before we got an **AUC = 0.8383** with baseline model, and when adding feature engineering, we got **AUC = 0.8446** 

our goal is to see feature engineering with cross validation can beat this or not

## 2. Rebuild feature engineering on X_train.

In [54]:
# feature engineering for churn dataset
def add_features(X:pd.DataFrame):
    # turn `tenure` column into buckets
    bin_edges = [0, 12, 24, 48, 72]
    bin_labels = ['0-12', '13-24', '25-48', '49-72']
    X['tenure_bucket'] = pd.cut(X['tenure'], bins=bin_edges, labels=bin_labels, include_lowest=True)

    # count the number of services available per customer
    services = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
    X["services_count"] = (X[services]=="Yes").sum(axis=1)

    # flag a new customer
    def check_new_customer(tenure):
        if tenure < 6:
            return 1
        else:
            return 0
    X["is_new_customer"] = X["tenure"].map(check_new_customer)

    # check for automatic payment
    def check_automatic_payment(payment_method):
        if "automatic" in payment_method:
            return 1
        else:
            return 0
    X["has_auto_payment"] = X["PaymentMethod"].map(check_automatic_payment)
    X["monthly_charge_per_tenure"] = (X["MonthlyCharges"]/(X["tenure"]+1))
    return X

X_train_features = add_features(X_train.copy())
X_test_features = add_features(X_test.copy())

## 3. Preprocessors

### 3.1 Baseline preprocessor (no engineered features)

In [57]:
num_cols = ["tenure", "MonthlyCharges"]
binary_cols = ["SeniorCitizen", "Partner", "Dependents", "PaperlessBilling"]
nominal_cols = ["InternetService", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV",
                "StreamingMovies", "Contract", "PaymentMethod"]

# binary map before implementing column transformer
for col in binary_cols:
    X_train[col] = X_train[col].map({"Yes": 1, "No": 0, "Male": 1, "Female": 0})

preprocessor_no_features = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("ohe", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), nominal_cols)
], remainder="passthrough")

### 3.2 Full preprocessor (with engineered features)

In [58]:
num_cols = ["tenure", "MonthlyCharges","services_count", "monthly_charge_per_tenure"]
binary_cols = ["SeniorCitizen", "Partner", "Dependents", "PaperlessBilling"]
nominal_cols = ["InternetService", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV",
                "StreamingMovies", "Contract", "PaymentMethod", "tenure_bucket"]

# binary map before implementing column transformer
for col in binary_cols:
    X_train_features[col] = X_train_features[col].map({"Yes": 1, "No": 0, "Male": 1, "Female": 0})

preprocessor_with_features = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("ohe", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), nominal_cols)
], remainder="passthrough")

## 4. CV setup

In [110]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(pipe, X, y, metric_name):
    scores = cross_val_score(estimator=pipe, X= X, y=y, scoring= metric_name, cv=skf)
    return f"ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}"

## 5. Run cross-validation

### 5.1 LogReg baseline (no features)

In [111]:
# build pipe with no feature engineering
pipe_no_features = Pipeline([
    ("prep", preprocessor_no_features),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

In [112]:
baseline_no_features_scores = run_cv(pipe=pipe_no_features, 
                                    X=X_train,
                                    y=y_train, metric_name="roc_auc")
print(baseline_no_features_scores)

ROC-AUC: 0.8442 ± 0.0118


### 5.2 LogReg baseline + features

In [ ]:
# build pipe with feature engineering
pipe_with_features = Pipeline([
    ("prep", preprocessor_with_features),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

In [137]:
baseline_with_features_scores = run_cv(pipe=pipe_with_features, 
                                    X=X_train_features,
                                    y=y_train, metric_name="roc_auc")
print(baseline_with_features_scores)

ROC-AUC: 0.8476 ± 0.0114


### 5.3 RandomForest + features

In [122]:
random_forrest = RandomForestClassifier(
    n_estimators=110, criterion = "entropy",
    max_depth= 8, n_jobs=-1, random_state=42
)
pipe_with_rf = Pipeline([
    ("prep", preprocessor_with_features),
    ("clf", random_forrest)
])

In [123]:
rf_with_features_scores = run_cv(pipe=pipe_with_rf, 
                                    X=X_train_features,
                                    y=y_train, metric_name="roc_auc")
print(rf_with_features_scores)

ROC-AUC: 0.8458 ± 0.0093


### 5.4 GradientBoosting  + features

In [126]:
gb_model = GradientBoostingClassifier(
    learning_rate=0.05, max_depth=3,
    n_estimators=130, random_state=42
)
pipe_with_gb = Pipeline([
    ("prep", preprocessor_with_features),
    ("clf", gb_model)
])

In [127]:
gb_with_features_scores = run_cv(pipe=pipe_with_gb, 
                                    X=X_train_features,
                                    y=y_train, metric_name="roc_auc")
print(gb_with_features_scores)

ROC-AUC: 0.8495 ± 0.0120


## 6. Results Table

<div align="center">

| Model | mean AUC | std  | n_folds |
|:-------|:-----:|:------:|:-----:|
| LogReg (no features)  | 0.8442 | 0.0114 | 5 |
| LogReg (with features) | **0.8476** | 0.0114 | 5 |
| RandomForest (with features)    | **0.8458** | 0.0093 | 5 |
| GradientBoosting (with features)  | **0.8495** | 0.0120 | 5 |

</div>

- hyperparameters of both RandomForestClassifier and GradientBoostingClassifier were coming from trying serval values of these hyperparameters and at the end i picked which score the best. and this means that these two models are kinda optimistic.
- what actually will give an honest indication from them is using **nested cross-validation**.
- for Logistic regression model, i used the default parameters.

## 7. Verdict

- from these results we can say that all models have very close scores in both **mean AUC** and **std**, so the decision of choosing the winner will be based on **Interpretability, Deployability, Training time, and Honesty.**
    - **Interpretability**: LogReg (with features) wins this, because you can understand why this result, why it gives this feature a small weight, etc.
    - **Deployability**: again LogReg (with features) wins this, small and fast model compared to complex models like (RF and GB)
    - **Training time**: again LogReg (with features) wins this. it's the fastest model in training.
    - **Honesty**: LogReg use default hyperparameters and on the other hand, RF and GB using hyperparameters that were tuned by trying and picking.

## 8. Decision: which model proceeds to 05

<details>
<summary>💡 <b>Winner: LogReg + features</b></summary>

there is no distinguish gap between the four models in the area of **AUC** and **std**, so the decision will be based on another vision.

LogReg is the most understandable, the lighter, the fastest, and reasonable without any fine tunning.

</details>
